In [1]:
!pip install -q transformers peft accelerate sentencepiece

In [2]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [3]:
import os
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from peft import PeftModel

BASE_MODEL_NAME = "google/mt5-large"

LORA_ADAPTER_DIR = "/content/drive/MyDrive/mt5_large_lora_r16_experiment_D_fixed/best_lora_adapter"

MERGED_MODEL_DIR = "/content/drive/MyDrive/mt5_large_lora_r16_experiment_D_fixed/merged_full_model"

os.makedirs(MERGED_MODEL_DIR, exist_ok=True)

device = "cuda" if torch.cuda.is_available() else "cpu"
USE_BF16 = torch.cuda.is_available() and torch.cuda.get_device_capability(0)[0] >= 8

print("Device:", device)
print("BF16:", USE_BF16)

Device: cuda
BF16: True


In [4]:
!pip uninstall torchao

Found existing installation: torchao 0.10.0
Uninstalling torchao-0.10.0:
  Would remove:
    /usr/local/lib/python3.12/dist-packages/torchao-0.10.0.dist-info/*
    /usr/local/lib/python3.12/dist-packages/torchao/*
Proceed (Y/n)? Y
  Successfully uninstalled torchao-0.10.0


In [5]:
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_NAME)

base_model = AutoModelForSeq2SeqLM.from_pretrained(
    BASE_MODEL_NAME,
    torch_dtype=torch.bfloat16 if USE_BF16 else torch.float32,
    device_map="auto"
)

model = PeftModel.from_pretrained(
    base_model,
    LORA_ADAPTER_DIR
)

model.eval()

print("LoRA adapter loaded successfully")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/642 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/376 [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/4.31M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/65.0 [00:00<?, ?B/s]

...

Loading weights:   0%|          | 0/560 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/4.92G [00:00<?, ?B/s]

The tied weights mapping and config for this model specifies to tie shared.weight to encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie shared.weight to decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

LoRA adapter loaded successfully


In [6]:
merged_model = model.merge_and_unload()

merged_model.save_pretrained(
    MERGED_MODEL_DIR,
    safe_serialization=True
)

tokenizer.save_pretrained(MERGED_MODEL_DIR)

print("Merged full mT5 model saved at:")
print(MERGED_MODEL_DIR)

/usr/local/lib/python3.12/dist-packages/peft/tuners/tuners_utils.py:683: UserWarning: Input and output embeddings are no longer tied after merging. Setting `tie_word_embeddings=False` in the model config.
  warnings.warn(


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Merged full mT5 model saved at:
/content/drive/MyDrive/mt5_large_lora_r16_experiment_D_fixed/merged_full_model
